# 01 — Data Exploration
Load both datasets, inspect, build corpus files at 1k / 5k / 20k docs.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [2]:
from data.loader import load_dataset, build_corpus
from config import SMOKE_CORPUS_SIZE, CORPUS_SIZES, CORPUS_DIR
import json, os

## Load FinQA

In [3]:
finqa_train = load_dataset('finqa', split='train', max_samples=500)
finqa_test  = load_dataset('finqa', split='test',  max_samples=200)
print(f'FinQA train: {len(finqa_train)} | test: {len(finqa_test)}')
print('Sample:', finqa_train[0])

FinQA train: 500 | test: 200
Sample: {'id': 'ADI/2009/page_49.pdf-1', 'question': 'what is the the interest expense in 2009?', 'answer': '3.8', 'answer_type': 'numeric', 'supporting_docs': [{'doc_id': 'ADI/2009/page_49.pdf-1', 'title': 'ADI/2009/page_49.pdf', 'text': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) . if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million . foreign currency exposure as more fully described in note 2i . in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s . dollar-based exposures by entering into forward foreign currency exchange contracts . the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months . currently , our largest foreign currency exposure is 

## Load MultiHop-RAG

In [4]:
multihop_train = load_dataset('multihop', split='train', max_samples=500)
print(f'MultiHop train: {len(multihop_train)}')
print('Sample:', multihop_train[0])

MultiHopRAG.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2556 [00:00<?, ? examples/s]

MultiHop train: 500
Sample: {'id': '', 'question': 'Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?', 'answer': 'Sam Bankman-Fried', 'answer_type': 'inference_query', 'supporting_docs': [], 'dataset': 'multihop'}


## Build Corpus at Multiple Sizes

In [5]:
# For large corpus sizes we need more samples
finqa_large    = load_dataset('finqa', split='train')
multihop_large = load_dataset('multihop', split='train')
all_samples    = finqa_large + multihop_large
print(f'Total samples: {len(all_samples)}')

Total samples: 8807


In [6]:
os.makedirs(CORPUS_DIR, exist_ok=True)
for size in [1000, 5000, 20000]:
    corpus = build_corpus(all_samples, corpus_size=size)
    path   = os.path.join(CORPUS_DIR, f'corpus_{size}.json')
    with open(path, 'w') as f:
        json.dump(corpus, f)
    print(f'Saved corpus_{size}.json — {len(corpus)} docs')

Saved corpus_1000.json — 1000 docs
Saved corpus_5000.json — 5000 docs
Saved corpus_20000.json — 6251 docs


## Chunking Stats

In [7]:
from pipeline.chunker import chunk_documents
import json

with open(os.path.join(CORPUS_DIR, 'corpus_1000.json')) as f:
    corpus_1k = json.load(f)

chunks = chunk_documents(corpus_1k)
print(f'1k docs -> {len(chunks)} chunks')
print(f'Sample chunk: {chunks[0]}')

1k docs -> 2658 chunks
Sample chunk: {'chunk_id': 'ADI/2009/page_49.pdf-1_c0', 'doc_id': 'ADI/2009/page_49.pdf-1', 'title': 'ADI/2009/page_49.pdf', 'text': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) . if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million . foreign currency exposure as more fully described in note 2i . in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s . dollar-based exposures by entering into forward foreign currency exchange contracts . the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months . currently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominat

## Embedding Shape Verification

In [8]:
from pipeline.embedder import embed_chunks
sample_chunks = chunks[:50]
emb = embed_chunks(sample_chunks)
print(f'Embeddings shape: {emb.shape}')  # expect (50, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (50, 384)


## Save Eval Question Samples

In [9]:
from config import EVAL_SAMPLE_SIZE, ACCURACY_DIR
os.makedirs(ACCURACY_DIR, exist_ok=True)

finqa_eval = finqa_test[:EVAL_SAMPLE_SIZE]
with open(os.path.join(ACCURACY_DIR, 'finqa_eval_questions.json'), 'w') as f:
    json.dump(finqa_eval, f)

multihop_eval = multihop_train[:EVAL_SAMPLE_SIZE]
with open(os.path.join(ACCURACY_DIR, 'multihop_eval_questions.json'), 'w') as f:
    json.dump(multihop_eval, f)

print(f'Saved {len(finqa_eval)} FinQA eval + {len(multihop_eval)} MultiHop eval questions')

Saved 200 FinQA eval + 200 MultiHop eval questions
